### Lahore Air Quality & Smog Forecast
#### Notebook 6: Admin Forecast Generation
Purpose

This notebook serves as the administrator interface for generating air quality forecasts. The latest environmental measurements will be manually entered, after which the trained machine learning models predict PM2.5 concentrations for the next three days. The predicted values are then converted into Air Quality Index (AQI) values, classified into AQI categories, assigned smog risk levels, and paired with health recommendations. The final forecast is prepared for manual publication on the project website.

In [8]:
import joblib
import pandas as pd
from datetime import datetime

In [9]:
model_day1 = joblib.load("pm25_day1_model.pkl")
model_day2 = joblib.load("pm25_day2_model.pkl")
model_day3 = joblib.load("pm25_day3_model.pkl")

Sample input

In [10]:
admin_input = {
    "pm25": 110,
    "temperature": 35,
    "relativehumidity": 48,
    "wind_speed": 12,
    "wind_direction": 220
}

In [11]:
input_df = pd.DataFrame([admin_input])

In [12]:
pred_day1 = model_day1.predict(input_df)[0]
pred_day2 = model_day2.predict(input_df)[0]
pred_day3 = model_day3.predict(input_df)[0]

In [13]:
def pm25_to_aqi(pm25):
    if pm25 <= 12.0:
        return round((50 / 12.0) * pm25)
    elif pm25 <= 35.4:
        return round(((100 - 51) / (35.4 - 12.1)) * (pm25 - 12.1) + 51)
    elif pm25 <= 55.4:
        return round(((150 - 101) / (55.4 - 35.5)) * (pm25 - 35.5) + 101)
    elif pm25 <= 150.4:
        return round(((200 - 151) / (150.4 - 55.5)) * (pm25 - 55.5) + 151)
    elif pm25 <= 250.4:
        return round(((300 - 201) / (250.4 - 150.5)) * (pm25 - 150.5) + 201)
    elif pm25 <= 350.4:
        return round(((400 - 301) / (350.4 - 250.5)) * (pm25 - 250.5) + 301)
    elif pm25 <= 500.4:
        return round(((500 - 401) / (500.4 - 350.5)) * (pm25 - 350.5) + 401)
    else:
        return 500

def get_aqi_category(aqi):
    if aqi <= 50:
        return "Good"
    elif aqi <= 100:
        return "Moderate"
    elif aqi <= 150:
        return "Unhealthy for Sensitive Groups"
    elif aqi <= 200:
        return "Unhealthy"
    elif aqi <= 300:
        return "Very Unhealthy"
    else:
        return "Hazardous"

def get_smog_risk(aqi):
    if aqi <= 50:
        return "Low 🟢"
    elif aqi <= 100:
        return "Mild 🟡"
    elif aqi <= 150:
        return "Moderate 🟠"
    elif aqi <= 200:
        return "High 🔴"
    elif aqi <= 300:
        return "Very High 🟣"
    else:
        return "Extreme ⚫"

def get_health_advice(aqi):
    if aqi <= 50:
        return "Air quality is good. Enjoy outdoor activities."
    elif aqi <= 100:
        return "Air quality is acceptable. Sensitive individuals should limit prolonged outdoor activity."
    elif aqi <= 150:
        return "Children, older adults, and people with heart or lung conditions should reduce outdoor activity."
    elif aqi <= 200:
        return "Avoid prolonged outdoor exercise. Wear a mask if spending extended time outside."
    elif aqi <= 300:
        return "Stay indoors where possible. Keep windows closed and use air filtration if available."
    else:
        return "Avoid going outdoors except when necessary. Follow official health advisories."

In [14]:
predictions = [pred_day1, pred_day2, pred_day3]
days = ["Today", "Tomorrow", "Day After Tomorrow"]

results = []

for day, pm25 in zip(days, predictions):
    aqi = pm25_to_aqi(pm25)
    results.append({
        "Day": day,
        "Predicted_PM25": round(pm25, 2),
        "AQI": aqi,
        "AQI_Category": get_aqi_category(aqi),
        "Smog_Risk": get_smog_risk(aqi),
        "Health_Advice": get_health_advice(aqi)
    })
results_df = pd.DataFrame(results)
results_df

,Day,Predicted_PM25,AQI,AQI_Category,Smog_Risk,Health_Advice
0,Today,96.31,172,Unhealthy,High 🔴,Avoid prolonged outdoor exercise. Wear a mask ...
1,Tomorrow,83.43,165,Unhealthy,High 🔴,Avoid prolonged outdoor exercise. Wear a mask ...
2,Day After Tomorrow,73.59,160,Unhealthy,High 🔴,Avoid prolonged outdoor exercise. Wear a mask ...


#### Reflection

In this notebook, I created the administrative workflow for generating air quality forecasts. The trained machine learning models predict PM2.5 concentrations, which are then converted into AQI values, categorized, and paired with smog risk levels and health recommendations. The final forecast is ready to be manually published on the project website for public access.
